<a href="https://colab.research.google.com/github/AzizulHakim00/DFU-ImageGuard/blob/main/notebooks/DFU_Reliable_V2_Rescue_Resume_OneCell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DFU Reliable V2 — Rescue and Resume
Preserves completed V2 trials, removes failed temporary checkpoint files, frees the duplicated active backup, and resumes the interrupted trial with a storage-bounded checkpoint writer.

In [ ]:
import os, sys, shutil, subprocess, json, time, uuid
from pathlib import Path
from google.colab import drive

MOUNT=Path('/content/drive'); MY=MOUNT/'MyDrive'
if not MY.is_dir(): drive.mount(str(MOUNT), force_remount=False)
if not MY.is_dir(): raise RuntimeError('Google Drive unavailable; rescue not started.')
probe=MY/'DFU-ImageGuard'/'_mount_verification'/f'rescue_{uuid.uuid4().hex}.txt'
probe.parent.mkdir(parents=True,exist_ok=True); value=str(time.time_ns()); probe.write_text(value)
if probe.read_text()!=value: raise RuntimeError('Drive write/read verification failed.')
probe.unlink(); print('Drive write/read verification: PASS')

subprocess.run([sys.executable,'-m','pip','install','-q','timm>=1.0.9','kagglehub>=0.3','ImageHash>=4.3','scikit-learn>=1.5','scipy>=1.13','matplotlib>=3.9','pandas>=2.2','Pillow>=10.4','tabulate>=0.9'],check=True)
REPO='https://github.com/AzizulHakim00/DFU-ImageGuard.git'
CODE_COMMIT='613f7150cb8957222a92ed2b41058780d7496435'
WORK=Path('/content/DFU-ImageGuard-v2-rescue')
if WORK.exists(): shutil.rmtree(WORK)
subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO,str(WORK)],check=True)
subprocess.run(['git','-C',str(WORK),'checkout',CODE_COMMIT],check=True)
os.chdir(WORK); sys.path.insert(0,str(WORK))
for module_name in list(sys.modules):
    if module_name=='src' or module_name.startswith('src.'):
        del sys.modules[module_name]
loaded=subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
if loaded!=CODE_COMMIT: raise RuntimeError(f'Commit mismatch: {loaded} != {CODE_COMMIT}')
print('Loaded V2 storage-rescue commit:',loaded)

from src import reliable_runner_v2 as runner
from src.reliable_storage_rescue import (
    ORIGINAL_TRAINING_COMMIT,
    install_rescue_patches,
    prepare_existing_v2_run,
)

audit=prepare_existing_v2_run(
    drive_root='/content/drive/MyDrive/DFU-ImageGuard',
    backup_root='/content/drive/MyDrive/DFU-ImageGuard-Backup',
    run_id='RELIABLE_DFU_CV_V2',
)
stale_active=Path('/content/drive/MyDrive/DFU-ImageGuard-Backup/runs/RELIABLE_DFU_CV_V2/_active_trial')
if stale_active.exists():
    released=sum(p.stat().st_size for p in stale_active.rglob('*') if p.is_file())
    shutil.rmtree(stale_active,ignore_errors=True)
    print('Released stale same-account active backup bytes:',released)
install_rescue_patches(runner)
print('Storage rescue patches: INSTALLED')
print('Resume action:',audit['resume_action'])
settings=runner.ReliableSettingsV2(
    run_id='RELIABLE_DFU_CV_V2',
    seeds=(2026,2027,2028),
    folds=(0,1,2,3,4),
    models=('convnextv2_tiny','mobilenetv3_large','densenet121'),
    max_epochs=30,
    patience=7,
    batch_size=16,
    num_workers=2,
    target_sensitivity=.95,
    source_commit=ORIGINAL_TRAINING_COMMIT,
)
result=runner.run_reliable_framework_v2(settings)
print(json.dumps(result,indent=2,default=str))


Mounted at /content/drive
Drive write/read verification: PASS
Loaded V2 storage-rescue commit: 613f7150cb8957222a92ed2b41058780d7496435
{
  "policy_version": 1,
  "run_id": "RELIABLE_DFU_CV_V2",
  "completed_trials": 6,
  "expected_trials": 45,
  "valid_incomplete_resume_count": 1,
  "valid_incomplete_resumes": [
    {
      "path": "/content/drive/MyDrive/DFU-ImageGuard/runs/RELIABLE_DFU_CV_V2/trials/convnextv2_tiny/seed_2028/fold_1/last_resume.pt",
      "bytes": 334657865,
      "sha256": "4fab2b04e067aa2a29fb068d65e2d5a3ddffd729910341405089fb623381ea26",
      "model_key": "convnextv2_tiny",
      "seed": 2028,
      "fold": 0,
      "epoch": 6,
      "source_commit": "349143b4d8b16f885adce3559542f6c202a2bca1"
    }
  ],
  "invalid_resumes": [],
  "restored_from_secondary": null,
  "resume_action": "resume_valid_incomplete_trial",
  "restart_cleanup": [],
  "removed_temporary_files": [],
  "removed_temporary_bytes": 0,
  "secondary_active_bytes_released": 0,
  "original_training_co

100%|██████████| 44.7M/44.7M [00:00<00:00, 176MB/s]


RESUME: completed trial skipped — convnextv2_tiny seed=2026 fold=1
SECONDARY BACKUP: VERIFIED [metadata] 49 file(s).
RESUME: completed trial skipped — mobilenetv3_large seed=2026 fold=1
SECONDARY BACKUP: VERIFIED [metadata] 49 file(s).
RESUME: completed trial skipped — densenet121 seed=2026 fold=1
SECONDARY BACKUP: VERIFIED [metadata] 49 file(s).
RESUME: completed trial skipped — convnextv2_tiny seed=2027 fold=1
SECONDARY BACKUP: VERIFIED [metadata] 49 file(s).
RESUME: completed trial skipped — mobilenetv3_large seed=2027 fold=1
SECONDARY BACKUP: VERIFIED [metadata] 49 file(s).
RESUME: completed trial skipped — densenet121 seed=2027 fold=1
SECONDARY BACKUP: VERIFIED [metadata] 49 file(s).


model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

mobilenetv3_large 2028 1 {'epoch': 1, 'train_loss': 0.666183914547882, 'selection_auc': 0.9991677070328756}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
mobilenetv3_large 2028 1 {'epoch': 2, 'train_loss': 0.14586257572832084, 'selection_auc': 0.9969482591205437}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
mobilenetv3_large 2028 1 {'epoch': 3, 'train_loss': 0.11775581134391841, 'selection_auc': 0.999445138021917}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
mobilenetv3_large 2028 1 {'epoch': 4, 'train_loss': 0.03486658669833838, 'selection_auc': 0.9997225690109585}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
mobilenetv3_large 2028 1 {'epoch': 5, 'train_loss': 0.018841839037263133, 'selection_auc': 1.0}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
mobilenetv3_large 2028 1 {'epoch': 6, 'train_loss': 0.03987748974334824, 'selection_auc': 0.9988902760438342}
SECONDARY BACKUP: VERI

model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

densenet121 2028 1 {'epoch': 1, 'train_loss': 0.3657608462899339, 'selection_auc': 0.9994451380219169}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
densenet121 2028 1 {'epoch': 2, 'train_loss': 0.09316642255175347, 'selection_auc': 0.999445138021917}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
densenet121 2028 1 {'epoch': 3, 'train_loss': 0.08335011602908957, 'selection_auc': 0.9997225690109586}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
densenet121 2028 1 {'epoch': 4, 'train_loss': 0.0295061150529221, 'selection_auc': 0.9994451380219169}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
densenet121 2028 1 {'epoch': 5, 'train_loss': 0.021976911167011542, 'selection_auc': 1.0}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 file(s).
densenet121 2028 1 {'epoch': 6, 'train_loss': 0.01612787029395501, 'selection_auc': 0.9999999999999999}
SECONDARY BACKUP: VERIFIED [active_trial_metadata_only] 2 